In [ ]:
# Design a Multi-Agent Workflow with LangGraph (25 Marks)
# 🧩 Scenario
# You are building an AI-powered customer support system for a fintech company.

# The system must handle:

# Transaction queries
# Fraud detection flags
# Refund requests
# General FAQs
# The company wants:

# High accuracy
# Step-by-step reasoning
# Ability to retry if answer is incorrect
# Modular, scalable architecture
# 💻 Task
# Design and implement a multi-agent workflow using LangGraph (or similar framework).

# ✅ 1. Agent Design
# Define at least 3 agents, such as:

# Retrieval Agent
# Reasoning/Answer Agent
# Validation Agent
# Explain briefly (in comments or code):

# Each agent’s role
# Input/output
# ✅ 2. Graph Workflow Implementation
# Write code or pseudo-code to:

# Define state
# Add nodes (agents)
# Define edges
# Implement conditional logic
# 👉 Must include:

# Retry loop if validation fails
# Clear start and end states
# ✅ 3. State Management
# Show how state evolves across steps:

# Query
# Context
# Intermediate reasoning
# Final answer
# Validation flag
# ✅ 4. Task Delegation & Communication
# Demonstrate:

# How agents pass information
# How decisions are made between agents
# 🎯 Expected Outcome
# A clear multi-step, graph-based agent system that:

# Handles complex queries
# Demonstrates reasoning + validation
# Uses proper orchestration

In [1]:
!pip install langgraph

In [3]:
# ============================================
# 🧠 MULTI-AGENT SYSTEM (FIXED)
# ============================================

from langgraph.graph import StateGraph
from typing import TypedDict, List

# ============================================
# 📦 STATE (STRICT STRUCTURE)
# ============================================
class GraphState(TypedDict):
    query: str
    context: List[str]
    reasoning: str
    answer: str
    valid: bool

# ============================================
# 📊 MOCK KNOWLEDGE BASE
# ============================================
knowledge_base = [
    "Refunds are processed within 5-7 business days.",
    "Fraud transactions must be reported within 24 hours.",
    "Transaction status can be checked using transaction ID.",
    "Customer support is available 24/7."
]

# ============================================
# 🤖 AGENT 1: RETRIEVAL
# ============================================
def retrieval_agent(state: GraphState):
    query = state.get("query", "").lower()   # ✅ safe access

    context = []
    for doc in knowledge_base:
        if any(word in doc.lower() for word in query.split()):
            context.append(doc)

    if not context:
        context = ["No relevant info found."]

    return {**state, "context": context}

# ============================================
# 🤖 AGENT 2: REASONING
# ============================================
def reasoning_agent(state: GraphState):
    query = state.get("query", "")
    context = state.get("context", [])

    reasoning = f"Analyzing query: {query}\nUsing context: {context}"

    if "refund" in query.lower():
        answer = "Your refund will be processed within 5-7 business days."
    elif "fraud" in query.lower():
        answer = "Report fraud within 24 hours to secure your account."
    elif "transaction" in query.lower():
        answer = "You can check transaction status using your transaction ID."
    else:
        answer = "Please contact support for more details."

    return {**state, "reasoning": reasoning, "answer": answer}

# ============================================
# 🤖 AGENT 3: VALIDATION
# ============================================
def validation_agent(state: GraphState):
    answer = state.get("answer", "").lower()
    context = " ".join(state.get("context", [])).lower()

    valid = any(word in context for word in answer.split())

    return {**state, "valid": valid}

# ============================================
# 🔀 BUILD GRAPH
# ============================================
def build_graph():
    builder = StateGraph(GraphState)

    builder.add_node("retrieve", retrieval_agent)
    builder.add_node("reason", reasoning_agent)
    builder.add_node("validate", validation_agent)

    builder.set_entry_point("retrieve")
    builder.add_edge("retrieve", "reason")
    builder.add_edge("reason", "validate")

    # Retry logic
    def check_validation(state: GraphState):
        return "retry" if not state.get("valid", False) else "end"

    builder.add_conditional_edges(
        "validate",
        check_validation,
        {
            "retry": "reason",
            "end": "__end__"
        }
    )

    return builder.compile()

# ============================================
# 🚀 MAIN
# ============================================
def main():
    graph = build_graph()

    queries = [
        "What is refund policy?",
        "How to report fraud?",
        "Check my transaction status"
    ]

    for q in queries:
        print("\n==============================")
        print("❓ Query:", q)

        # ✅ Proper initial state
        state = {
            "query": q,
            "context": [],
            "reasoning": "",
            "answer": "",
            "valid": False
        }

        result = graph.invoke(state)

        print("🧠 Reasoning:\n", result["reasoning"])
        print("✅ Answer:", result["answer"])
        print("✔ Validation:", result["valid"])

# ============================================
# ▶️ RUN
# ============================================
if __name__ == "__main__":
    main()


❓ Query: What is refund policy?
🧠 Reasoning:
 Analyzing query: What is refund policy?
Using context: ['Refunds are processed within 5-7 business days.', 'Customer support is available 24/7.']
✅ Answer: Your refund will be processed within 5-7 business days.
✔ Validation: True

❓ Query: How to report fraud?
🧠 Reasoning:
 Analyzing query: How to report fraud?
Using context: ['Fraud transactions must be reported within 24 hours.', 'Customer support is available 24/7.']
✅ Answer: Report fraud within 24 hours to secure your account.
✔ Validation: True

❓ Query: Check my transaction status
🧠 Reasoning:
 Analyzing query: Check my transaction status
Using context: ['Fraud transactions must be reported within 24 hours.', 'Transaction status can be checked using transaction ID.']
✅ Answer: You can check transaction status using your transaction ID.
✔ Validation: True
